In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

In [8]:
minute_df = pd.read_csv("~/Desktop/data/spx_2013_2024_1m.csv", parse_dates=['date'])
minute_df = minute_df.set_index('date').sort_index()
print(minute_df.head(3))
print(f"分钟数据区间: {minute_df.index.min()} -> {minute_df.index.max()}")
print(f"交易日数量: {minute_df.index.normalize().nunique()}")

                        open     high      low    close  volume
date                                                           
2013-01-02 09:30:00  1426.19  1443.44  1426.19  1443.44     0.0
2013-01-02 09:31:00  1443.71  1448.55  1443.71  1448.55     0.0
2013-01-02 09:32:00  1448.93  1450.62  1448.83  1450.51     0.0
分钟数据区间: 2013-01-02 09:30:00 -> 2024-03-28 15:59:00
交易日数量: 2829


In [9]:
# -------------------- 1. 按交易日构建特征与标签 --------------------
def build_daily_dataset(minute_df, horizon_days=3):
    """从分钟数据构造按交易日对齐的中周期特征，并预测未来3个交易日方向。"""
    minute_df = minute_df.copy()
    minute_df['trading_day'] = minute_df.index.normalize()
    minute_df['minute_return'] = minute_df.groupby('trading_day')['close'].pct_change()

    daily_df = minute_df.groupby('trading_day').agg(
        open=('open', 'first'),
        high=('high', 'max'),
        low=('low', 'min'),
        close=('close', 'last'),
        volume=('volume', 'sum'),
        intraday_vol_1d=('minute_return', 'std'),
        bars_in_day=('close', 'size')
    )

    # 1. 中周期收益率特征
    daily_df['return_1d'] = daily_df['close'].pct_change(1)
    daily_df['return_3d'] = daily_df['close'].pct_change(3)
    daily_df['return_5d'] = daily_df['close'].pct_change(5)
    daily_df['return_10d'] = daily_df['close'].pct_change(10)
    daily_df['return_20d'] = daily_df['close'].pct_change(20)
    daily_df['return_60d'] = daily_df['close'].pct_change(60)

    # 2. 日内/隔夜波动切换
    daily_df['overnight_gap_1d'] = daily_df['open'] / daily_df['close'].shift(1) - 1
    daily_df['intraday_return_1d'] = daily_df['close'] / daily_df['open'] - 1
    gap_abs_5d = daily_df['overnight_gap_1d'].abs().rolling(5).mean().replace(0, np.nan)
    daily_df['intraday_to_overnight_vol_5d'] = daily_df['intraday_vol_1d'].rolling(5).mean() / gap_abs_5d
    daily_df['overnight_gap_5d_mean'] = daily_df['overnight_gap_1d'].rolling(5).mean()
    daily_df['intraday_return_5d_mean'] = daily_df['intraday_return_1d'].rolling(5).mean()

    # 3. 最近几天高低点结构
    daily_df['range_pct_1d'] = (daily_df['high'] - daily_df['low']) / daily_df['close']
    daily_df['range_pct_5d'] = daily_df['range_pct_1d'].rolling(5).mean()
    daily_df['close_to_5d_high'] = daily_df['close'] / daily_df['high'].rolling(5).max() - 1
    daily_df['close_to_5d_low'] = daily_df['close'] / daily_df['low'].rolling(5).min() - 1
    daily_df['close_to_10d_high'] = daily_df['close'] / daily_df['high'].rolling(10).max() - 1
    daily_df['close_to_10d_low'] = daily_df['close'] / daily_df['low'].rolling(10).min() - 1
    daily_df['close_to_20d_high'] = daily_df['close'] / daily_df['high'].rolling(20).max() - 1
    daily_df['close_to_20d_low'] = daily_df['close'] / daily_df['low'].rolling(20).min() - 1
    daily_df['close_to_60d_high'] = daily_df['close'] / daily_df['high'].rolling(60).max() - 1
    daily_df['close_to_60d_low'] = daily_df['close'] / daily_df['low'].rolling(60).min() - 1

    # 4. 相对周/月/季均线位置和趋势结构
    sma_5d = daily_df['close'].rolling(5).mean()
    sma_20d = daily_df['close'].rolling(20).mean()
    sma_60d = daily_df['close'].rolling(60).mean()
    daily_df['dist_to_SMA_5d'] = daily_df['close'] / sma_5d - 1
    daily_df['dist_to_SMA_20d'] = daily_df['close'] / sma_20d - 1
    daily_df['dist_to_SMA_60d'] = daily_df['close'] / sma_60d - 1
    daily_df['sma_spread_5d_20d'] = sma_5d / sma_20d - 1
    daily_df['sma_spread_20d_60d'] = sma_20d / sma_60d - 1

    # 5. 波动 regime 和趋势稳定性
    daily_df['volatility_5d'] = daily_df['return_1d'].rolling(5).std()
    daily_df['volatility_20d'] = daily_df['return_1d'].rolling(20).std()
    daily_df['volatility_60d'] = daily_df['return_1d'].rolling(60).std()
    daily_df['vol_regime_5d_vs_20d'] = daily_df['volatility_5d'] / daily_df['volatility_20d'].replace(0, np.nan)
    daily_df['vol_regime_20d_vs_60d'] = daily_df['volatility_20d'] / daily_df['volatility_60d'].replace(0, np.nan)
    daily_df['positive_day_ratio_5d'] = (daily_df['return_1d'] > 0).rolling(5).mean()
    daily_df['positive_day_ratio_20d'] = (daily_df['return_1d'] > 0).rolling(20).mean()
    daily_df['rolling_sharpe_20d'] = daily_df['return_1d'].rolling(20).mean() / daily_df['volatility_20d'].replace(0, np.nan)
    daily_df['bars_in_day_ratio_20d'] = daily_df['bars_in_day'] / daily_df['bars_in_day'].rolling(20).mean() - 1

    # 6. 目标变量：未来3个交易日收盘相对当前收盘是否上涨
    daily_df['future_return_3d'] = daily_df['close'].shift(-horizon_days) / daily_df['close'] - 1
    daily_df['Target'] = (daily_df['future_return_3d'] > 0).astype(int)

    daily_df = daily_df.dropna().copy()
    print("目标分布:")
    print(daily_df['Target'].value_counts(normalize=True).rename('ratio'))

    return daily_df

horizon_days = 3
df = build_daily_dataset(minute_df, horizon_days=horizon_days)
FEATURES = [
    col for col in df.columns
    if col not in ['open', 'high', 'low', 'close', 'volume', 'future_return_3d', 'Target']
]
TARGET = 'Target'

目标分布:
Target
1    0.593637
0    0.406363
Name: ratio, dtype: float64


In [10]:
# 检查最终特征列表
print(f"样本数量: {len(df)} 个交易日")
print(f"特征数量: {len(FEATURES)}")
print(FEATURES)

样本数量: 2766 个交易日
特征数量: 37
['intraday_vol_1d', 'bars_in_day', 'return_1d', 'return_3d', 'return_5d', 'return_10d', 'return_20d', 'return_60d', 'overnight_gap_1d', 'intraday_return_1d', 'intraday_to_overnight_vol_5d', 'overnight_gap_5d_mean', 'intraday_return_5d_mean', 'range_pct_1d', 'range_pct_5d', 'close_to_5d_high', 'close_to_5d_low', 'close_to_10d_high', 'close_to_10d_low', 'close_to_20d_high', 'close_to_20d_low', 'close_to_60d_high', 'close_to_60d_low', 'dist_to_SMA_5d', 'dist_to_SMA_20d', 'dist_to_SMA_60d', 'sma_spread_5d_20d', 'sma_spread_20d_60d', 'volatility_5d', 'volatility_20d', 'volatility_60d', 'vol_regime_5d_vs_20d', 'vol_regime_20d_vs_60d', 'positive_day_ratio_5d', 'positive_day_ratio_20d', 'rolling_sharpe_20d', 'bars_in_day_ratio_20d']


In [11]:
# -------------------- 2. Walk-forward 参数 --------------------

# 使用固定长度滚动训练窗口，更贴近真实的定期重训场景
train_window_days = 756   # 约 3 个交易年
test_window_days = 20     # 约 1 个交易月
embargo_days = horizon_days

fold_starts = list(range(train_window_days + embargo_days, len(df), test_window_days))

print(f"训练窗口: {train_window_days} 个交易日")
print(f"测试窗口: {test_window_days} 个交易日")
print(f"Embargo days: {embargo_days}")
print(f"Walk-forward folds: {len(fold_starts)}")
print(f"首个 fold 测试开始日: {df.index[fold_starts[0]].date()}")
print(f"最后一个 fold 测试开始日: {df.index[fold_starts[-1]].date()}")

训练窗口: 756 个交易日
测试窗口: 20 个交易日
Embargo days: 3
Walk-forward folds: 101
首个 fold 测试开始日: 2016-04-05
最后一个 fold 测试开始日: 2024-03-15


In [12]:
# -------------------- 3. Walk-forward 训练与评估 --------------------

model_params = dict(
    objective='binary:logistic',
    n_estimators=300,
    eval_metric='logloss',
    colsample_bytree=0.8,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

fold_metrics = []
oos_predictions = []
feature_importance_list = []

print("\n开始 walk-forward 训练...")

for fold_id, test_start in enumerate(fold_starts, start=1):
    train_end = test_start - embargo_days
    train_start = max(0, train_end - train_window_days)
    test_end = min(test_start + test_window_days, len(df))

    X_train = df[FEATURES].iloc[train_start:train_end]
    y_train = df[TARGET].iloc[train_start:train_end]
    X_test = df[FEATURES].iloc[test_start:test_end]
    y_test = df[TARGET].iloc[test_start:test_end]

    model = xgb.XGBClassifier(**model_params)
    model.fit(X_train, y_train, verbose=False)

    prediction_proba = model.predict_proba(X_test)[:, 1]
    predictions = (prediction_proba >= 0.5).astype(int)
    always_up_predictions = np.ones(len(y_test), dtype=int)

    fold_auc = roc_auc_score(y_test, prediction_proba) if y_test.nunique() > 1 else np.nan
    fold_accuracy = accuracy_score(y_test, predictions)
    fold_baseline_accuracy = accuracy_score(y_test, always_up_predictions)

    fold_metrics.append({
        'fold': fold_id,
        'train_start': X_train.index.min(),
        'train_end': X_train.index.max(),
        'test_start': X_test.index.min(),
        'test_end': X_test.index.max(),
        'rows_test': len(X_test),
        'accuracy': fold_accuracy,
        'auc': fold_auc,
        'always_up_accuracy': fold_baseline_accuracy,
        'excess_accuracy': fold_accuracy - fold_baseline_accuracy
    })

    oos_predictions.append(pd.DataFrame({
        'Target': y_test,
        'prediction': predictions,
        'prediction_proba': prediction_proba,
        'future_return_3d': df['future_return_3d'].iloc[test_start:test_end]
    }, index=X_test.index))

    feature_importance_list.append(pd.Series(model.feature_importances_, index=FEATURES, name=fold_id))

walk_forward_predictions = pd.concat(oos_predictions).sort_index()
fold_metrics_df = pd.DataFrame(fold_metrics)
mean_importances = pd.concat(feature_importance_list, axis=1).mean(axis=1).sort_values(ascending=False)

# -------------------- 4. Walk-forward 汇总结果 --------------------

wf_accuracy = accuracy_score(walk_forward_predictions['Target'], walk_forward_predictions['prediction'])
wf_auc = roc_auc_score(walk_forward_predictions['Target'], walk_forward_predictions['prediction_proba'])
wf_always_up_accuracy = accuracy_score(
    walk_forward_predictions['Target'],
    np.ones(len(walk_forward_predictions), dtype=int)
)
wf_excess_accuracy = wf_accuracy - wf_always_up_accuracy

print("walk-forward 训练完成。")
print(f"\n总 OOS 样本数: {len(walk_forward_predictions)}")
print(f"Walk-forward Accuracy: {wf_accuracy*100:.2f}%")
print(f"Walk-forward AUC: {wf_auc:.4f}")
print(f"永远预测上涨的基线准确率: {wf_always_up_accuracy*100:.2f}%")
print(f"相对基线的准确率超额: {wf_excess_accuracy*100:.2f}%")
print(f"OOS 上涨比例: {walk_forward_predictions['Target'].mean():.4f}")
print("分类报告:")
print(classification_report(walk_forward_predictions['Target'], walk_forward_predictions['prediction']))

print("\nFold 指标摘要:")
print(fold_metrics_df[['accuracy', 'auc', 'always_up_accuracy', 'excess_accuracy']].describe())

# -------------------- 5. 平均特征重要性分析 --------------------
print("\n平均特征重要性:")
print(mean_importances.head(20))


开始 walk-forward 训练...
walk-forward 训练完成。

总 OOS 样本数: 2007
Walk-forward Accuracy: 53.16%
Walk-forward AUC: 0.5029
永远预测上涨的基线准确率: 59.44%
相对基线的准确率超额: -6.28%
OOS 上涨比例: 0.5944
分类报告:
              precision    recall  f1-score   support

           0       0.41      0.35      0.38       814
           1       0.60      0.66      0.62      1193

    accuracy                           0.53      2007
   macro avg       0.50      0.50      0.50      2007
weighted avg       0.52      0.53      0.52      2007


Fold 指标摘要:
         accuracy         auc  always_up_accuracy  excess_accuracy
count  101.000000  101.000000          101.000000       101.000000
mean     0.532815    0.546783            0.595191        -0.062376
std      0.136094    0.172466            0.154620         0.157703
min      0.250000    0.125000            0.200000        -0.450000
25%      0.450000    0.414141            0.500000        -0.150000
50%      0.550000    0.560000            0.600000        -0.050000
75%      0.6000